# Preprocessing Dataset CSE-CIC-IDS2018

**Proyek:** Sistem Deteksi Intrusi Berbasis Machine Learning  
**Dataset:** CSE-CIC-IDS2018 (Communications Security Establishment & Canadian Institute for Cybersecurity)  
**Output:** `dataset_split.joblib`

---

## Prinsip Desain

1. **Netral antar model** — tidak ada feature selection berbasis algoritma tertentu. Ketiga model menerima input identik sehingga perbandingan bersifat *fair*.
2. **Anti-leakage** — semua keputusan yang bergantung pada distribusi data (varians, korelasi, scaling) hanya di-*fit* dari data training.
3. **Berbasis data aktual** — setiap keputusan penanganan (inf, duplikat, dll.) diambil setelah audit distribusi label.
4. **Mendukung target sistem** — F1 ≥ 85%, FPR ≤ 2%.

---

## Keputusan Berbasis Data Aktual

| Tahap | Keputusan | Justifikasi dari Data |
|---|---|---|
| Penanganan inf | Cap P99 | 98,6% dari 95.760 baris inf berasal dari Benign — replace max finite akan menaikkan FPR |
| Dedup | Hanya kelas Benign | Dedup penuh menghancurkan FTP-BruteForce (99,97% hilang) dan SlowHTTPTest (99,96% hilang) karena serangan ini repetitif by design |
| Stratified split | Berdasarkan label multiclass | SQL Injection hanya 87 sampel — stratify binary bisa menghasilkan distribusi 87-0-0 |
| Scaling | QuantileTransformer uniform | std Bwd Pkts/s = 92.686 (outlier ekstrem) — MinMax/Standard tidak robust |

---
## Sel 1 — Instalasi dan Impor Library

In [ ]:
!pip install polars --quiet

import polars as pl
import numpy as np
import pandas as pd
import os, gc, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.utils import class_weight
from itertools import combinations

print("Polars versi:", pl.__version__)

dataset_path = "/content/drive/MyDrive/Colab Notebooks"

csv_files = [
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv",
]

RANDOM_STATE = 42
print(f"Total file CSV: {len(csv_files)}")

Polars versi: 1.35.2
Total file CSV: 10


---
## Sel 2 — Inspeksi Schema Per File

In [ ]:
columns_per_file = {}
for f in csv_files:
    path = os.path.join(dataset_path, f)
    lf   = pl.scan_csv(path, n_rows=0)
    cols = [c.strip() for c in lf.collect_schema().names()]
    columns_per_file[f] = cols
    print(f"{f}: {len(cols)} kolom")

common_cols = set(columns_per_file[csv_files[0]])
for f in csv_files[1:]:
    common_cols &= set(columns_per_file[f])
common_cols = sorted(common_cols)
print(f"\nKolom umum di semua file: {len(common_cols)}")

Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 84 kolom
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 80 kolom

Kolom umum di semua file: 80


---
## Sel 3 — Penghapusan Kolom Identitas (Anti-Label Leakage)

In [ ]:
identity_cols = ['Dst Port', 'Protocol', 'Timestamp',
                 'Flow ID', 'Src IP', 'Dst IP', 'Src Port']
to_remove = [c for c in identity_cols if c in common_cols]
keep_cols = [c for c in common_cols if c not in to_remove]

if 'Label' not in keep_cols:
    keep_cols.append('Label')

print(f"Kolom identitas yang dihapus : {to_remove}")
print(f"Kolom yang dipertahankan     : {len(keep_cols)} (termasuk Label)")

Kolom identitas yang dihapus : ['Dst Port', 'Protocol', 'Timestamp']
Kolom yang dipertahankan     : 77 (termasuk Label)


---
## Sel 4 — Pemuatan Data CSV (String-First Strategy)

In [ ]:
lazy_frames = []
for f in csv_files:
    path = os.path.join(dataset_path, f)
    print(f"Memuat {f}...")
    lf = pl.scan_csv(path, infer_schema_length=0, ignore_errors=False)

    rename_map = {c: c.strip() for c in lf.collect_schema().names()}
    lf = lf.rename(rename_map)

    available = [c for c in keep_cols if c in lf.collect_schema().names()]
    lf = lf.select(available)

    if 'Label' in available:
        lf = lf.filter(pl.col("Label") != "Label")
        lf = lf.with_columns(
            pl.col("Label").str.replace("Infilteration", "Infiltration")
        )

    lazy_frames.append(lf)
    gc.collect()

print("\nSemua lazy frame siap.")

Memuat Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Friday-16-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Friday-23-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv...
Memuat Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv...
Memuat Friday-02-03-2018_TrafficForML_CICFlowMeter.csv...

Semua lazy frame siap.


---
## Sel 5 — Casting, Penggabungan, dan Verifikasi Data

In [ ]:
non_label_cols = [c for c in keep_cols if c != 'Label']

casted_frames = []
for lf in lazy_frames:
    cols_in_frame = lf.collect_schema().names()
    lf = lf.with_columns([
        pl.col(c).cast(pl.Float32, strict=False)
        for c in non_label_cols if c in cols_in_frame
    ])
    casted_frames.append(lf)

print("Casting selesai. Menggabungkan dan mengumpulkan data...")
full_lf  = pl.concat(casted_frames)
final_df = full_lf.collect(engine="streaming")

del lazy_frames, casted_frames, full_lf
gc.collect()

print(f"Shape data  : {final_df.shape}")
print(f"Memori      : {final_df.estimated_size() / 1024**2:.1f} MB")
print("\nDistribusi Label :")
print(final_df.group_by('Label').agg(pl.len()).sort('len', descending=True))

Casting selesai. Menggabungkan dan mengumpulkan data...
Shape data  : (16232943, 77)
Memori      : 4825.0 MB

Distribusi Label :
shape: (15, 2)
┌────────────────────────┬──────────┐
│ Label                  ┆ len      │
│ ---                    ┆ ---      │
│ str                    ┆ u32      │
╞════════════════════════╪══════════╡
│ Benign                 ┆ 13484708 │
│ DDOS attack-HOIC       ┆ 686012   │
│ DDoS attacks-LOIC-HTTP ┆ 576191   │
│ DoS attacks-Hulk       ┆ 461912   │
│ Bot                    ┆ 286191   │
│ …                      ┆ …        │
│ DoS attacks-Slowloris  ┆ 10990    │
│ DDOS attack-LOIC-UDP   ┆ 1730     │
│ Brute Force -Web       ┆ 611      │
│ Brute Force -XSS       ┆ 230      │
│ SQL Injection          ┆ 87       │
└────────────────────────┴──────────┘


---
## Sel 6 — Audit Nilai Tak Hingga (Inf)

In [ ]:
float_cols = [c for c in final_df.columns if final_df[c].dtype == pl.Float32]

print("=== AUDIT NILAI INF ===")
inf_cols = []
for c in float_cols:
    cnt = final_df.filter(pl.col(c).is_infinite()).height
    if cnt > 0:
        inf_cols.append(c)
        print(f"  {c}: {cnt:,} baris")

if inf_cols:
    inf_mask   = pl.any_horizontal([pl.col(c).is_infinite() for c in inf_cols])
    inf_rows   = final_df.filter(inf_mask)

    print(f"\nTotal baris unik dengan nilai inf : {inf_rows.height:,}")
    print(f"Persentase dari total             : {100*inf_rows.height/final_df.height:.3f}%")
    print("\nDistribusi label pada baris inf:")
    print(inf_rows.group_by('Label').agg(pl.len()).sort('len', descending=True))

    n_inf_benign = inf_rows.filter(pl.col('Label').str.to_lowercase() == 'benign').height
    n_inf_attack = inf_rows.height - n_inf_benign
    pct_attack   = 100 * n_inf_attack / inf_rows.height

    print(f"\nBaris inf yang merupakan ATTACK  : {n_inf_attack:,} ({pct_attack:.1f}%)")
    print(f"Baris inf yang merupakan BENIGN  : {n_inf_benign:,} ({100-pct_attack:.1f}%)")
    del inf_rows
else:
    print("Tidak ada nilai inf.")

=== AUDIT NILAI INF ===
  Flow Byts/s: 36,039 baris
  Flow Pkts/s: 95,760 baris

Total baris unik dengan nilai inf : 95,760
Persentase dari total             : 0.590%

Distribusi label pada baris inf:
shape: (3, 2)
┌────────────────┬───────┐
│ Label          ┆ len   │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ Benign         ┆ 94459 │
│ Infiltration   ┆ 1295  │
│ FTP-BruteForce ┆ 6     │
└────────────────┴───────┘

Baris inf yang merupakan ATTACK  : 1,301 (1.4%)
Baris inf yang merupakan BENIGN  : 94,459 (98.6%)


---
## Sel 7 — Penanganan Nilai Tak Hingga: Cap Persentil ke-99

In [ ]:
if inf_cols:
    print("=== PENANGANAN INF: CAP PERSENTIL KE-99 ===")
    for c in inf_cols:
        p99   = final_df.filter(~pl.col(c).is_infinite())[c].quantile(0.99)
        n_inf = final_df.filter(pl.col(c).is_infinite()).height
        final_df = final_df.with_columns(
            pl.when(pl.col(c).is_infinite())
              .then(p99)
              .otherwise(pl.col(c))
              .alias(c)
        )
        print(f"  {c}: {n_inf:,} nilai inf -> P99 = {p99:,.4f}")

    inf_sisa = sum(
        final_df.filter(pl.col(c).is_infinite()).height for c in inf_cols
    )
    print(f"\nVerifikasi inf tersisa : {inf_sisa} (harus 0)")

null_total = final_df.null_count().sum_horizontal().item()
print(f"Total null             : {null_total} (harus 0)")
print(f"Shape                  : {final_df.shape}")

=== PENANGANAN INF: CAP PERSENTIL KE-99 ===
  Flow Byts/s: 36,039 nilai inf -> P99 = 1,291,666.6250
  Flow Pkts/s: 95,760 nilai inf -> P99 = 2,000,000.0000

Verifikasi inf tersisa : 0 (harus 0)
Total null             : 0 (harus 0)
Shape                  : (16232943, 77)


---
## Sel 8 — Penghapusan Nilai Negatif pada Fitur Non-Negatif

In [ ]:
nonneg_cols = [
    'Flow Duration',
    'Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Tot',   'Fwd IAT Mean', 'Fwd IAT Max', 'Fwd IAT Min',
    'Bwd IAT Tot',   'Bwd IAT Mean', 'Bwd IAT Max', 'Bwd IAT Min',
]
nonneg_cols = [c for c in nonneg_cols if c in final_df.columns]

neg_mask  = pl.any_horizontal([pl.col(c) < 0 for c in nonneg_cols])
neg_count = final_df.filter(neg_mask).height
print(f"Baris dengan nilai negatif: {neg_count}")

if neg_count > 0:
    print("Distribusi label pada baris negatif:")
    print(final_df.filter(neg_mask).group_by('Label').agg(pl.len()))
    final_df = final_df.filter(~neg_mask)
    print(f"Shape setelah penghapusan: {final_df.shape}")
else:
    print("Tidak ditemukan nilai negatif. ")

Baris dengan nilai negatif: 15
Distribusi label pada baris negatif:
shape: (1, 2)
┌────────┬─────┐
│ Label  ┆ len │
│ ---    ┆ --- │
│ str    ┆ u32 │
╞════════╪═════╡
│ Benign ┆ 15  │
└────────┴─────┘
Shape setelah penghapusan: (16232928, 77)


---
## Sel 9 — Penanganan Sentinel Value -1 pada Init Win Byts

In [ ]:
for c in ['Init Fwd Win Byts', 'Init Bwd Win Byts']:
    if c in final_df.columns:
        n_minus1 = final_df.filter(pl.col(c) == -1).height
        final_df = final_df.with_columns(
            pl.when(pl.col(c) == -1).then(0).otherwise(pl.col(c)).alias(c)
        )
        print(f"{c}: {n_minus1:,} sentinel -1 -> 0")

Init Fwd Win Byts: 4,432,594 sentinel -1 -> 0
Init Bwd Win Byts: 8,255,535 sentinel -1 -> 0


---
## Sel 10 — Penghapusan Duplikat Selektif (Hanya Kelas Benign)

In [ ]:
print("=== DISTRIBUSI SEBELUM DEDUP ===")
dist_before = final_df.group_by('Label').agg(
    pl.len().alias('count')
).sort('count', descending=True)
dist_before_dict = {r['Label']: r['count'] for r in dist_before.iter_rows(named=True)}
print(dist_before)

benign_df = final_df.filter(pl.col('Label').str.to_lowercase() == 'benign')
attack_df = final_df.filter(pl.col('Label').str.to_lowercase() != 'benign')

print(f"\nBenign sebelum dedup   : {benign_df.height:,}")
print(f"Attack (tidak di-dedup): {attack_df.height:,}")

benign_deduped  = benign_df.unique()
pct_benign_dup  = 100 * (benign_df.height - benign_deduped.height) / benign_df.height
print(f"Benign setelah dedup   : {benign_deduped.height:,}")
print(f"Duplikat Benign hapus  : {benign_df.height - benign_deduped.height:,} ({pct_benign_dup:.1f}%)")

final_df = pl.concat([benign_deduped, attack_df])
del benign_df, benign_deduped, attack_df
gc.collect()

print(f"\nShape setelah dedup selektif: {final_df.shape}")
print("\nDistribusi setelah dedup:")
print(final_df.group_by('Label').agg(pl.len()).sort('len', descending=True))

=== DISTRIBUSI SEBELUM DEDUP ===
shape: (15, 2)
┌────────────────────────┬──────────┐
│ Label                  ┆ count    │
│ ---                    ┆ ---      │
│ str                    ┆ u32      │
╞════════════════════════╪══════════╡
│ Benign                 ┆ 13484693 │
│ DDOS attack-HOIC       ┆ 686012   │
│ DDoS attacks-LOIC-HTTP ┆ 576191   │
│ DoS attacks-Hulk       ┆ 461912   │
│ Bot                    ┆ 286191   │
│ …                      ┆ …        │
│ DoS attacks-Slowloris  ┆ 10990    │
│ DDOS attack-LOIC-UDP   ┆ 1730     │
│ Brute Force -Web       ┆ 611      │
│ Brute Force -XSS       ┆ 230      │
│ SQL Injection          ┆ 87       │
└────────────────────────┴──────────┘

Benign sebelum dedup   : 13,484,693
Attack (tidak di-dedup): 2,748,235
Benign setelah dedup   : 8,986,829
Duplikat Benign hapus  : 4,497,864 (33.4%)

Shape setelah dedup selektif: (11735064, 77)

Distribusi setelah dedup:
shape: (15, 2)
┌────────────────────────┬─────────┐
│ Label                  ┆ len 

---
## Sel 11 — Pembuatan Label Binary dan Checkpoint Parquet

In [ ]:
final_df = final_df.with_columns(
    pl.when(pl.col("Label").str.to_lowercase() == "benign")
      .then(0).otherwise(1).cast(pl.Int8).alias("attack")
)

n_benign = final_df.filter(pl.col('attack') == 0).height
n_attack = final_df.filter(pl.col('attack') == 1).height
print("Distribusi label binary:")
print(final_df.group_by('attack').agg(pl.len()).sort('attack'))
print(f"Rasio Benign:Attack = {n_benign/n_attack:.2f}:1")

checkpoint_path = os.path.join(dataset_path, "cicids2018_cleaned.parquet")
final_df.write_parquet(checkpoint_path, compression="snappy")
print(f"\nCheckpoint disimpan : {checkpoint_path}")
print(f"Shape               : {final_df.shape}")

Distribusi label binary:
shape: (2, 2)
┌────────┬─────────┐
│ attack ┆ len     │
│ ---    ┆ ---     │
│ i8     ┆ u32     │
╞════════╪═════════╡
│ 0      ┆ 8986829 │
│ 1      ┆ 2748235 │
└────────┴─────────┘
Rasio Benign:Attack = 3.27:1

Checkpoint disimpan : /content/drive/MyDrive/Colab Notebooks/cicids2018_cleaned.parquet
Shape               : (11735064, 78)


---
## Sel 12 — Pembagian Data: Train / Validasi / Test (70:15:15)

In [ ]:
print("Memuat checkpoint...")
df         = pl.read_parquet(checkpoint_path)
label_multi = df['Label'].to_pandas()
y_binary   = df['attack'].to_pandas()
X          = df.drop(['Label', 'attack']).to_pandas()
del df, final_df
gc.collect()

print("Membagi data 70% train — 30% ...")
X_train, X_temp, y_train, y_temp, lbl_train, lbl_temp = train_test_split(
    X, y_binary, label_multi,
    test_size=0.30, stratify=label_multi, random_state=RANDOM_STATE
)
del X, y_binary, label_multi
gc.collect()

print("Membagi 30% menjadi 15% validasi — 15% test...")
X_val, X_test, y_val, y_test, lbl_val, lbl_test = train_test_split(
    X_temp, y_temp, lbl_temp,
    test_size=0.50, stratify=lbl_temp, random_state=RANDOM_STATE
)
del X_temp, y_temp, lbl_temp
gc.collect()

total = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]
print(f"\nTraining set   : {X_train.shape[0]:,} baris ({X_train.shape[0]/total*100:.0f}%)")
print(f"Validation set : {X_val.shape[0]:,} baris ({X_val.shape[0]/total*100:.0f}%)")
print(f"Test set       : {X_test.shape[0]:,} baris ({X_test.shape[0]/total*100:.0f}%)")

print("\nDistribusi per jenis serangan pada Test Set:")
print(lbl_test.value_counts().sort_index().to_string())

Memuat checkpoint...
Membagi data 70% train — 30% ...
Membagi 30% menjadi 15% validasi — 15% test...

Training set   : 8,214,544 baris (70%)
Validation set : 1,760,260 baris (15%)
Test set       : 1,760,260 baris (15%)

Distribusi per jenis serangan pada Test Set:
Label
Benign                      1348024
Bot                           42928
Brute Force -Web                 92
Brute Force -XSS                 35
DDOS attack-HOIC             102902
DDOS attack-LOIC-UDP            260
DDoS attacks-LOIC-HTTP        86428
DoS attacks-GoldenEye          6227
DoS attacks-Hulk              69287
DoS attacks-SlowHTTPTest      20983
DoS attacks-Slowloris          1648
FTP-BruteForce                29004
Infiltration                  24290
SQL Injection                    13
SSH-Bruteforce                28139


---
## Sel 13 — Penghapusan Fitur Zero Variance

In [ ]:
variances = X_train.var(axis=0)
zero_var  = variances[variances == 0].index.tolist()

print(f"Fitur zero-variance pada training set ({len(zero_var)}):")
for c in sorted(zero_var):
    print(f"  - {c}")

if zero_var:
    X_train = X_train.drop(columns=zero_var)
    X_val   = X_val.drop(columns=zero_var)
    X_test  = X_test.drop(columns=zero_var)
    print(f"\nShape training set : {X_train.shape}")

Fitur zero-variance pada training set (8):
  - Bwd Blk Rate Avg
  - Bwd Byts/b Avg
  - Bwd PSH Flags
  - Bwd Pkts/b Avg
  - Bwd URG Flags
  - Fwd Blk Rate Avg
  - Fwd Byts/b Avg
  - Fwd Pkts/b Avg

Shape training set : (8214544, 68)


---
## Sel 14 — Penghapusan Fitur yang Identik Secara Statistik

In [ ]:
print("Mencari pasangan fitur yang identik pada training set...")
cols   = X_train.columns.tolist()
sample = X_train.sample(n=min(100_000, len(X_train)), random_state=RANDOM_STATE)

identical_pairs   = []
to_drop_identical = set()

for i, j in combinations(range(len(cols)), 2):
    c1, c2 = cols[i], cols[j]
    if c2 in to_drop_identical:
        continue
    if sample[c1].equals(sample[c2]):
        if X_train[c1].equals(X_train[c2]):
            identical_pairs.append((c1, c2))
            to_drop_identical.add(c2)

print(f"Pasangan fitur identik ditemukan ({len(identical_pairs)}):")
for c1, c2 in identical_pairs:
    print(f"  {c1}  ==  {c2}  -> hapus: {c2}")

to_drop_identical = list(to_drop_identical)
if to_drop_identical:
    X_train = X_train.drop(columns=to_drop_identical)
    X_val   = X_val.drop(columns=to_drop_identical)
    X_test  = X_test.drop(columns=to_drop_identical)
    print(f"\nShape training set : {X_train.shape}")

del sample
gc.collect()

Mencari pasangan fitur yang identik pada training set...
Pasangan fitur identik ditemukan (7):
  Bwd Pkt Len Mean  ==  Bwd Seg Size Avg  -> hapus: Bwd Seg Size Avg
  CWE Flag Count  ==  Fwd URG Flags  -> hapus: Fwd URG Flags
  Fwd PSH Flags  ==  SYN Flag Cnt  -> hapus: SYN Flag Cnt
  Fwd Pkt Len Mean  ==  Fwd Seg Size Avg  -> hapus: Fwd Seg Size Avg
  Subflow Bwd Pkts  ==  Tot Bwd Pkts  -> hapus: Tot Bwd Pkts
  Subflow Fwd Byts  ==  TotLen Fwd Pkts  -> hapus: TotLen Fwd Pkts
  Subflow Fwd Pkts  ==  Tot Fwd Pkts  -> hapus: Tot Fwd Pkts

Shape training set : (8214544, 61)


0

---
## Sel 15 — Penghapusan Fitur Berkorelasi Tinggi (Threshold r > 0,92)

In [ ]:
print("Menghitung matriks korelasi pada training set ...")

corr_sample_size = min(2_000_000, len(X_train))
corr_sample = X_train.sample(n=corr_sample_size, random_state=RANDOM_STATE)
corr  = corr_sample.corr().abs()
del corr_sample
gc.collect()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

threshold    = 0.92
to_drop_corr = set()
pairs_found  = []

for c1 in upper.columns:
    for c2 in upper.index:
        val = upper.loc[c2, c1]
        if pd.isna(val) or val <= threshold:
            continue
        if c1 in to_drop_corr or c2 in to_drop_corr:
            continue
        avg1     = corr[c1].drop(c1).mean()
        avg2     = corr[c2].drop(c2).mean()
        drop_col = c1 if avg1 < avg2 else c2
        to_drop_corr.add(drop_col)
        pairs_found.append((c1, c2, val, drop_col))

print(f"Pasangan berkorelasi tinggi (r > {threshold}): {len(pairs_found)}")
print(f"\n{'Fitur 1':<25} {'Fitur 2':<25} {'r':>6}  Dihapus")
print("-" * 70)
for c1, c2, val, dropped in sorted(pairs_found, key=lambda x: -x[2]):
    print(f"{c1:<25} {c2:<25} {val:>6.3f}  {dropped}")

to_drop_corr = list(to_drop_corr)
print(f"\nTotal fitur dihapus: {len(to_drop_corr)}")

if to_drop_corr:
    X_train = X_train.drop(columns=to_drop_corr)
    X_val   = X_val.drop(columns=to_drop_corr)
    X_test  = X_test.drop(columns=to_drop_corr)
    print(f"Shape training set : {X_train.shape}")

with open(os.path.join(dataset_path, "dropped_features_correlation.txt"), "w") as f:
    f.write("\n".join(sorted(to_drop_corr)))

del corr, upper
gc.collect()

print(f"\nFitur final ({X_train.shape[1]} fitur):")
for i, c in enumerate(sorted(X_train.columns), 1):
    print(f"  {i:2d}. {c}")

Menghitung matriks korelasi pada training set ...
Pasangan berkorelasi tinggi (r > 0.92): 23

Fitur 1                   Fitur 2                        r  Dihapus
----------------------------------------------------------------------
RST Flag Cnt              ECE Flag Cnt               1.000  ECE Flag Cnt
Subflow Bwd Pkts          Bwd Header Len             0.999  Subflow Bwd Pkts
Subflow Fwd Pkts          Fwd Header Len             0.998  Subflow Fwd Pkts
TotLen Bwd Pkts           Bwd Header Len             0.997  TotLen Bwd Pkts
Subflow Bwd Byts          Bwd Header Len             0.997  Subflow Bwd Byts
Fwd IAT Tot               Flow Duration              0.996  Fwd IAT Tot
Pkt Size Avg              Pkt Len Mean               0.996  Pkt Size Avg
Idle Mean                 Idle Max                   0.995  Idle Mean
Fwd Header Len            Fwd Act Data Pkts          0.995  Fwd Act Data Pkts
Flow IAT Min              Flow IAT Mean              0.991  Flow IAT Mean
Fwd IAT Max         

---
## Sel 16 — Normalisasi Fitur: QuantileTransformer

In [ ]:
print("=== VERIFIKASI DAN PEMBERSIHAN NaN ===")
nan_train = X_train.isna().sum().sum()
nan_val   = X_val.isna().sum().sum()
nan_test  = X_test.isna().sum().sum()
print(f"NaN — Train: {nan_train:,} | Val: {nan_val:,} | Test: {nan_test:,}")

if nan_train > 0 or nan_val > 0 or nan_test > 0:
    X_train   = X_train.dropna()
    y_train   = y_train.loc[X_train.index]
    lbl_train = lbl_train.loc[X_train.index]

    X_val   = X_val.dropna()
    y_val   = y_val.loc[X_val.index]
    lbl_val = lbl_val.loc[X_val.index]

    X_test   = X_test.dropna()
    y_test   = y_test.loc[X_test.index]
    lbl_test = lbl_test.loc[X_test.index]

    print(f"Shape setelah drop NaN:")
    print(f"  Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
    print(f"  NaN tersisa: {X_train.isna().sum().sum()} | Inf: {np.isinf(X_train.values).sum()}")

print("\n=== NORMALISASI: QUANTILE TRANSFORMER ===")
sample_size = min(200_000, len(X_train))
print(f"Stratified sampling {sample_size:,} baris untuk fitting...")

_, fit_sample = train_test_split(
    X_train, test_size=sample_size,
    stratify=lbl_train, random_state=RANDOM_STATE
)

qt = QuantileTransformer(
    output_distribution='uniform',
    n_quantiles=10_000,
    subsample=sample_size,
    random_state=RANDOM_STATE
)
qt.fit(fit_sample)
del fit_sample
gc.collect()

print("Mentransformasi training set...")
X_train_scaled = qt.transform(X_train).astype(np.float32)
print("Mentransformasi validation set...")
X_val_scaled   = qt.transform(X_val).astype(np.float32)
print("Mentransformasi test set...")
X_test_scaled  = qt.transform(X_test).astype(np.float32)

joblib.dump(qt, os.path.join(dataset_path, "quantile_transformer.pkl"))

print(f"\nShape scaled  : {X_train_scaled.shape}")
print(f"Range         : [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")
print(f"Mean          : {X_train_scaled.mean():.4f}")
print(f"Std           : {X_train_scaled.std():.4f}")
print(f"NaN di scaled : {np.isnan(X_train_scaled).sum()}")

=== VERIFIKASI DAN PEMBERSIHAN NaN ===
NaN — Train: 1,874 | Val: 373 | Test: 412
Shape setelah drop NaN:
  Train: (8212670, 38) | Val: (1759887, 38) | Test: (1759848, 38)
  NaN tersisa: 0 | Inf: 0

=== NORMALISASI: QUANTILE TRANSFORMER ===
Stratified sampling 200,000 baris untuk fitting...
Mentransformasi training set...
Mentransformasi validation set...
Mentransformasi test set...

Shape scaled  : (8212670, 38)
Range         : [0.0000, 1.0000]
Mean          : 0.3297
Std           : 0.3734
NaN di scaled : 0


---
## Sel 17 — Penghitungan Bobot Kelas

In [ ]:
weights          = class_weight.compute_class_weight(
    'balanced', classes=np.unique(y_train.values), y=y_train.values
)
class_weights    = {0: float(weights[0]), 1: float(weights[1])}
n_benign_train   = (y_train.values == 0).sum()
n_attack_train   = (y_train.values == 1).sum()
scale_pos_weight = float(n_benign_train / n_attack_train)

print(f"Jumlah Benign (train)    : {n_benign_train:,}")
print(f"Jumlah Attack (train)    : {n_attack_train:,}")
print(f"Rasio Benign:Attack      : {scale_pos_weight:.4f}:1")
print(f"\nclass_weights (MLP/AE)   : {class_weights}")
print(f"scale_pos_weight (LGBM)  : {scale_pos_weight:.4f}")

Jumlah Benign (train)    : 6,289,506
Jumlah Attack (train)    : 1,923,164
Rasio Benign:Attack      : 3.2704:1

class_weights (MLP/AE)   : {0: 0.6528867291008229, 1: 2.135197518256373}
scale_pos_weight (LGBM)  : 3.2704


---
## Sel 18 — Penyimpanan Artifact

In [ ]:
dataset_split = {
    'X_train'          : X_train_scaled,
    'y_train'          : y_train.values,
    'X_val'            : X_val_scaled,
    'y_val'            : y_val.values,
    'X_test'           : X_test_scaled,
    'y_test'           : y_test.values,
    'label_train'      : lbl_train.values,
    'label_val'        : lbl_val.values,
    'label_test'       : lbl_test.values,
    'features'         : X_train.columns.tolist(),
    'n_features'       : X_train.shape[1],
    'class_weights'    : class_weights,
    'scale_pos_weight' : scale_pos_weight,
}

output_file = os.path.join(dataset_path, "dataset_split.joblib")
joblib.dump(dataset_split, output_file, compress=3)
print(f"Tersimpan : {output_file}")

with open(os.path.join(dataset_path, "selected_features.txt"), "w") as f:
    f.write("\n".join(X_train.columns.tolist()))
print("Tersimpan : selected_features.txt")

Tersimpan : /content/drive/MyDrive/Colab Notebooks/dataset_split.joblib
Tersimpan : selected_features.txt


---
## Sel 19 — Ringkasan dan Validasi Kesiapan Data

In [ ]:
print("=" * 65)
print(" PREPROCESSING SELESAI — CSE-CIC-IDS2018")
print("=" * 65)

print(f"\n{'Parameter':<35} {'Nilai'}")
print("-" * 55)
print(f"{'Jumlah fitur final':<35} {X_train.shape[1]}")
print(f"{'Training set':<35} {X_train_scaled.shape[0]:,} baris")
print(f"{'Validation set':<35} {X_val_scaled.shape[0]:,} baris")
print(f"{'Test set':<35} {X_test_scaled.shape[0]:,} baris")
print(f"{'Rasio Benign:Attack (train)':<35} {n_benign_train/n_attack_train:.2f}:1")
print(f"{'class_weights Benign':<35} {class_weights[0]:.4f}")
print(f"{'class_weights Attack':<35} {class_weights[1]:.4f}")
print(f"{'scale_pos_weight (LightGBM)':<35} {scale_pos_weight:.4f}")

print("\n--- Validasi Target Sistem ---")
total_benign_test = (y_test.values == 0).sum()
total_attack_test = (y_test.values == 1).sum()
max_fp = int(total_benign_test * 0.02)
print(f"Target F1 >= 85% (binary):")
print(f"  Sampel Attack di test : {total_attack_test:,}")
print(f"  Sampel Benign di test : {total_benign_test:,}")

print("\n--- Kesiapan Evaluasi Per Jenis Serangan (Test Set) ---")
print(f"  {'Jenis Serangan':<35} {'Sampel':>8}  Status")
print("  " + "-" * 58)
for lbl_name, cnt in sorted(lbl_test.value_counts().items()):
    status = "Cukup" if cnt >= 30 else "< 30 sampel"
    print(f"  {lbl_name:<35} {cnt:>8,}  {status}")

print("\n--- Verifikasi Kualitas Data Akhir ---")
print(f"  NaN di X_train_scaled : {np.isnan(X_train_scaled).sum()}")
print(f"  NaN di X_val_scaled   : {np.isnan(X_val_scaled).sum()}")
print(f"  NaN di X_test_scaled  : {np.isnan(X_test_scaled).sum()}")
print(f"  Range X_train_scaled  : [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]")

print("\n--- File yang Disimpan ---")
for fname in [
    "dataset_split.joblib",
    "quantile_transformer.pkl",
    "cicids2018_cleaned.parquet",
    "selected_features.txt",
    "dropped_features_correlation.txt",
]:
    print(f"  v  {fname}")

 PREPROCESSING SELESAI — CSE-CIC-IDS2018

Parameter                           Nilai
-------------------------------------------------------
Jumlah fitur final                  38
Training set                        8,212,670 baris
Validation set                      1,759,887 baris
Test set                            1,759,848 baris
Rasio Benign:Attack (train)         3.27:1
class_weights Benign                0.6529
class_weights Attack                2.1352
scale_pos_weight (LightGBM)         3.2704

--- Validasi Target Sistem ---
Target F1 >= 85% (binary):
  Sampel Attack di test : 412,126
  Sampel Benign di test : 1,347,722

--- Kesiapan Evaluasi Per Jenis Serangan (Test Set) ---
  Jenis Serangan                        Sampel  Status
  ----------------------------------------------------------
  Benign                              1,347,722  Cukup
  Bot                                   42,928  Cukup
  Brute Force -Web                          92  Cukup
  Brute Force -XSS          